# SBERT STS-B Fine-Tuning (no NLI) — 10 Seed Experiment

This notebook runs the **SBERT-STSb-base** fine-tuning experiment across **10 random seeds**.
Unlike the previous version of this notebook, it does **not** load an NLI-trained checkpoint -
it starts every seed from plain pretrained `bert-base-uncased` and fine-tunes directly on STS-B.
This targets the paper's `SBERT-STSb-base` row (Table 2, "Trained on STS benchmark dataset"
section), not `SBERT-NLI-STSb-base`.

### Purpose

The Sentence-BERT paper reports:

- **SBERT-STSb-base:** **84.67 ± 0.19** Spearman on STS-B (trained on STS-B only, no NLI).
- The paper reports results averaged over **10 random seeds**, 4 epochs.
- STS-B fine-tuning uses the regression objective: cosine similarity between the two sentence
  embeddings, optimized with MSE.

This notebook repeats that experiment structure for 10 seeds and reports:

- test Spearman for every seed
- mean
- standard deviation
- minimum / maximum
- difference from the paper's reported mean of 84.67

### Important implementation detail

Every seed starts from **plain pretrained `bert-base-uncased`** (via `AutoModel.from_pretrained`),
not any NLI-trained checkpoint. This is a **10-seed reproduction of the direct STS-B fine-tuning
setup**, matching Table 2's naming convention (no "NLI" in the model name means no NLI
pre-training stage - see Table 2's caption: "SBERT was fine-tuned on the STSb dataset,
SBERT-NLI was pretrained on the NLI datasets, then fine-tuned on the STSb dataset").

Hardware adaptation retained from the single-seed notebook:

- physical batch size = 4
- gradient accumulation = 4
- effective batch size = 16
- max sequence length = 128
- AMP enabled on CUDA

The paper explicitly reports 4 epochs for the STS-B fine-tuning experiment, while some other
STS-B hyperparameters are not specified in the paper text; those implementation choices are
kept consistent with the previous notebook.


In [1]:
# ============================================================
# 1. IMPORTS
# ============================================================

import os
import math
import random
import time
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModel
from scipy.stats import spearmanr
from tqdm.auto import tqdm

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [2]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

MODEL_NAME = "bert-base-uncased"

def resolve_sts_dir():
    candidates = [
        Path("../data") / "STS",
        Path("data") / "STS",
        Path("/kaggle/input/datasets/adityammantri/data-table2/data/STS"),
    ]

    for c in candidates:
        if (c / "STSb").exists():
            return c

    # fallback: bounded recursive search under /kaggle/input
    root = Path("/kaggle/input")
    if root.exists():
        frontier = [(root, 0)]
        while frontier:
            current, depth = frontier.pop(0)
            try:
                children = [c for c in current.iterdir() if c.is_dir()]
            except PermissionError:
                continue
            if (current / "STSb").exists():
                return current
            if depth < 4:
                frontier.extend((c, depth + 1) for c in children)

    raise FileNotFoundError(
        "Could not locate a directory containing 'STSb'.\n"
        "Checked:\n" + "\n".join(f"  - {c}" for c in candidates)
    )


STS_DIR = resolve_sts_dir()
print("Resolved STS dir:", STS_DIR.resolve())

# Separate output directory so this experiment does not
# overwrite the NLI-init or single-seed experiments.
OUTPUT_DIR = Path("outputs") / "sbert_stsb_base_10_seeds"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED_CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
SEED_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = OUTPUT_DIR / "10_seed_results.csv"

# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

EPOCHS = 4
LEARNING_RATE = 2e-5

MAX_LENGTH = 128

PHYSICAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4

EFFECTIVE_BATCH_SIZE = (
    PHYSICAL_BATCH_SIZE *
    GRADIENT_ACCUMULATION_STEPS
)

EVAL_BATCH_SIZE = 32

# The paper reports 10 random seeds.
SEEDS = list(range(42, 52))

# Target: SBERT-STSb-base (Table 2), NOT SBERT-NLI-STSb-base (85.35).
PAPER_SCORE = 84.67

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

USE_AMP = torch.cuda.is_available()

print("Model:", MODEL_NAME)
print("Device:", DEVICE)

print("\nEpochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)

print("\nPhysical batch size:", PHYSICAL_BATCH_SIZE)
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("Effective batch size:", EFFECTIVE_BATCH_SIZE)

print("\nMax length:", MAX_LENGTH)
print("Seeds:", SEEDS)
print("Number of seeds:", len(SEEDS))
print("Paper score (SBERT-STSb-base):", PAPER_SCORE)

Resolved STS dir: /kaggle/input/datasets/adityammantri/data-table2/data/STS
Model: bert-base-uncased
Device: cuda

Epochs: 4
Learning rate: 2e-05

Physical batch size: 4
Gradient accumulation: 4
Effective batch size: 16

Max length: 128
Seeds: [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
Number of seeds: 10
Paper score (SBERT-STSb-base): 84.67


In [3]:
# ============================================================
# 3. REPRODUCIBILITY
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Do not force fully deterministic CUDA kernels because
    # that can reduce performance and is not required by the
    # paper's "10 random seeds" protocol.
    torch.backends.cudnn.benchmark = False


print("Seed function ready.")

Seed function ready.


In [4]:
# ============================================================
# 4. LOAD STS-B
# ============================================================

STSB_PATH = STS_DIR / "STSb"

if not STSB_PATH.exists():
    raise FileNotFoundError(
        f"STS-B dataset not found:\n{STSB_PATH.resolve()}"
    )

stsb_dataset = load_from_disk(STSB_PATH)

print(stsb_dataset)

for split in stsb_dataset.keys():
    print(
        f"{split:12s}: "
        f"{len(stsb_dataset[split]):,} examples"
    )

DatasetDict({
    train: Dataset({
        features: ['split', 'genre', 'dataset', 'year', 'sid', 'score', 'sentence1', 'sentence2'],
        num_rows: 5749
    })
    validation: Dataset({
        features: ['split', 'genre', 'dataset', 'year', 'sid', 'score', 'sentence1', 'sentence2'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['split', 'genre', 'dataset', 'year', 'sid', 'score', 'sentence1', 'sentence2'],
        num_rows: 1379
    })
})
train       : 5,749 examples
validation  : 1,500 examples
test        : 1,379 examples


In [5]:
# ============================================================
# 5. CONVERT STS-B SPLITS
# ============================================================

train_df = stsb_dataset["train"].to_pandas()
dev_df = stsb_dataset["validation"].to_pandas()
test_df = stsb_dataset["test"].to_pandas()

print("Train:", train_df.shape)
print("Dev:", dev_df.shape)
print("Test:", test_df.shape)

print("\nColumns:", train_df.columns.tolist())

Train: (5749, 8)
Dev: (1500, 8)
Test: (1379, 8)

Columns: ['split', 'genre', 'dataset', 'year', 'sid', 'score', 'sentence1', 'sentence2']


In [6]:
# ============================================================
# 6. LOAD TOKENIZER
# ============================================================
# No NLI checkpoint here - this experiment starts every seed from
# plain pretrained bert-base-uncased (see build_model() below).

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded:", MODEL_NAME)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded: bert-base-uncased


In [7]:
# ============================================================
# 7. STS-B DATASET
# ============================================================

class STSBDataset(torch.utils.data.Dataset):

    def __init__(self, dataframe):
        self.sentence1 = dataframe["sentence1"].astype(str).tolist()
        self.sentence2 = dataframe["sentence2"].astype(str).tolist()
        self.scores = dataframe["score"].astype(float).tolist()

    def __len__(self):
        return len(self.scores)

    def __getitem__(self, idx):
        return {
            "sentence1": self.sentence1[idx],
            "sentence2": self.sentence2[idx],
            "score": self.scores[idx]
        }


train_dataset = STSBDataset(train_df)
dev_dataset = STSBDataset(dev_df)
test_dataset = STSBDataset(test_df)

print("Train:", len(train_dataset))
print("Dev:", len(dev_dataset))
print("Test:", len(test_dataset))

Train: 5749
Dev: 1500
Test: 1379


In [8]:
# ============================================================
# 8. COLLATE FUNCTION
# ============================================================

def collate_fn(batch):

    sentences1 = [
        item["sentence1"]
        for item in batch
    ]

    sentences2 = [
        item["sentence2"]
        for item in batch
    ]

    scores = torch.tensor(
        [
            item["score"]
            for item in batch
        ],
        dtype=torch.float
    )

    inputs1 = tokenizer(
        sentences1,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    inputs2 = tokenizer(
        sentences2,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    return {
        "sentence1": inputs1,
        "sentence2": inputs2,
        "scores": scores
    }


print("Collate function ready.")

Collate function ready.


In [9]:
# ============================================================
# 9. MODEL DEFINITION
# ============================================================

class SBERTSTSRegressor(nn.Module):

    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder

    def mean_pooling(
        self,
        last_hidden_state,
        attention_mask
    ):
        mask = attention_mask.unsqueeze(-1).expand(
            last_hidden_state.size()
        ).float()

        summed = torch.sum(
            last_hidden_state * mask,
            dim=1
        )

        counts = torch.clamp(
            mask.sum(dim=1),
            min=1e-9
        )

        return summed / counts

    def encode(
        self,
        input_ids,
        attention_mask
    ):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        return self.mean_pooling(
            outputs.last_hidden_state,
            attention_mask
        )

    def forward(
        self,
        sentence1_inputs,
        sentence2_inputs
    ):
        u = self.encode(
            sentence1_inputs["input_ids"],
            sentence1_inputs["attention_mask"]
        )

        v = self.encode(
            sentence2_inputs["input_ids"],
            sentence2_inputs["attention_mask"]
        )

        return torch.nn.functional.cosine_similarity(
            u,
            v,
            dim=1
        )


print("Model definition ready.")

Model definition ready.


In [10]:
# ============================================================
# 10. SCORE NORMALIZATION
# ============================================================

def normalize_stsb_scores(scores):
    """
    STS-B labels are in [0, 5].
    Convert them to [0, 1] for the cosine-regression target.
    """
    return scores / 5.0


print("STS-B score normalization: [0, 5] -> [0, 1]")

STS-B score normalization: [0, 5] -> [0, 1]


In [11]:
# ============================================================
# 11. MODEL FACTORY
# ============================================================

def build_model():
    """
    Build a fresh, plain pretrained BERT encoder for each seed.

    No NLI checkpoint is loaded here - this targets SBERT-STSb-base
    (direct fine-tuning on STS-B only), not SBERT-NLI-STSb-base.
    Every seed starts from the same pretrained bert-base-uncased
    weights; the seed changes stochastic fine-tuning behavior
    (data shuffling order, dropout masks, etc.).
    """

    encoder = AutoModel.from_pretrained(MODEL_NAME)

    model = SBERTSTSRegressor(
        encoder
    ).to(DEVICE)

    return model


print("Model factory ready (plain pretrained BERT, no NLI init).")


Model factory ready (plain pretrained BERT, no NLI init).


In [12]:
# ============================================================
# 12. DATALOADERS
# ============================================================

def build_dataloaders(seed):

    # Explicit generator makes the training shuffle seed
    # reproducible for each experiment.
    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=PHYSICAL_BATCH_SIZE,
        shuffle=True,
        generator=generator,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    dev_loader = DataLoader(
        dev_dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    return train_loader, dev_loader, test_loader


print("Dataloader factory ready.")

Dataloader factory ready.


In [13]:
# ============================================================
# 13. TRAINING FUNCTION
# ============================================================

def train_one_epoch(
    model,
    dataloader,
    optimizer,
    criterion,
    scaler,
    device,
    accumulation_steps
):

    model.train()

    total_loss = 0.0

    optimizer.zero_grad(set_to_none=True)

    progress_bar = tqdm(
        dataloader,
        desc="Training",
        leave=False
    )

    for step, batch in enumerate(progress_bar):

        inputs1 = {
            key: value.to(device)
            for key, value in batch["sentence1"].items()
        }

        inputs2 = {
            key: value.to(device)
            for key, value in batch["sentence2"].items()
        }

        scores = normalize_stsb_scores(
            batch["scores"]
        ).to(device)

        with torch.amp.autocast(
            device_type="cuda",
            enabled=USE_AMP
        ):
            predictions = model(
                inputs1,
                inputs2
            )

        # Loss computed in fp32, outside autocast - consistent with
        # the other notebooks in this project (a margin/regression
        # loss on fp16 cosine similarities is numerically fragile).
        loss = criterion(
            predictions.float(),
            scores
        )

        loss = loss / accumulation_steps

        scaler.scale(loss).backward()

        should_update = (
            (step + 1) % accumulation_steps == 0
            or (step + 1) == len(dataloader)
        )

        if should_update:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item() * accumulation_steps

        progress_bar.set_postfix(
            loss=f"{loss.item() * accumulation_steps:.4f}"
        )

    return total_loss / len(dataloader)


In [14]:
# ============================================================
# 14. VALIDATION / TEST EVALUATION
# ============================================================

@torch.no_grad()
def evaluate_stsb(
    model,
    dataloader,
    device
):

    model.eval()

    predictions = []
    gold_scores = []

    for batch in tqdm(
        dataloader,
        desc="Evaluating",
        leave=False
    ):

        inputs1 = {
            key: value.to(device)
            for key, value in batch["sentence1"].items()
        }

        inputs2 = {
            key: value.to(device)
            for key, value in batch["sentence2"].items()
        }

        scores = batch["scores"].numpy()

        with torch.amp.autocast(
            device_type="cuda",
            enabled=USE_AMP
        ):
            similarity = model(
                inputs1,
                inputs2
            )

        similarity = (
            similarity
            .float()
            .cpu()
            .numpy()
        )

        predictions.extend(
            similarity.tolist()
        )

        # Dividing by 5 does not change Spearman because it
        # preserves the ranking of the gold labels.
        gold_scores.extend(
            (scores / 5.0).tolist()
        )

    predictions = np.array(predictions)
    gold_scores = np.array(gold_scores)

    rho = spearmanr(
        predictions,
        gold_scores
    ).statistic

    return rho * 100

In [15]:
# ============================================================
# 15. SINGLE-SEED EXPERIMENT
# ============================================================

def run_seed(seed):

    print("\n" + "=" * 80)
    print(f"STARTING SEED {seed}")
    print("=" * 80)

    set_seed(seed)

    train_loader, dev_loader, test_loader = build_dataloaders(seed)

    model = build_model()

    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP
    )

    best_dev_score = -float("inf")
    best_epoch = None
    history = []

    seed_checkpoint_path = (
        SEED_CHECKPOINT_DIR /
        f"sbert_stsb_base_seed_{seed}.pt"
    )

    start_time = time.time()

    for epoch in range(EPOCHS):

        print("\n" + "-" * 70)
        print(f"Seed {seed} | Epoch {epoch + 1}/{EPOCHS}")
        print("-" * 70)

        train_loss = train_one_epoch(
            model=model,
            dataloader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            scaler=scaler,
            device=DEVICE,
            accumulation_steps=GRADIENT_ACCUMULATION_STEPS
        )

        dev_spearman = evaluate_stsb(
            model=model,
            dataloader=dev_loader,
            device=DEVICE
        )

        history.append({
            "seed": seed,
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "dev_spearman": dev_spearman
        })

        print(f"Train loss: {train_loss:.6f}")
        print(f"Dev Spearman: {dev_spearman:.4f}")

        if dev_spearman > best_dev_score:

            best_dev_score = dev_spearman
            best_epoch = epoch + 1

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "model_name": MODEL_NAME,
                    "pooling": "mean",
                    "max_length": MAX_LENGTH,
                    "learning_rate": LEARNING_RATE,
                    "epochs": EPOCHS,
                    "seed": seed,
                    "best_epoch": best_epoch,
                    "best_dev_spearman": best_dev_score,
                },
                seed_checkpoint_path
            )

            print(
                f"Best checkpoint saved "
                f"(epoch {best_epoch})"
            )

    # --------------------------------------------------------
    # Load best checkpoint for this seed
    # --------------------------------------------------------

    best_checkpoint = torch.load(
        seed_checkpoint_path,
        map_location=DEVICE,
        weights_only=False
    )

    model.load_state_dict(
        best_checkpoint["model_state_dict"]
    )

    model.eval()

    test_spearman = evaluate_stsb(
        model=model,
        dataloader=test_loader,
        device=DEVICE
    )

    elapsed_minutes = (
        time.time() - start_time
    ) / 60.0

    result = {
        "seed": seed,
        "best_epoch": best_epoch,
        "best_dev_spearman": best_dev_score,
        "test_spearman": test_spearman,
        "difference_from_paper": test_spearman - PAPER_SCORE,
        "runtime_minutes": elapsed_minutes
    }

    history_df = pd.DataFrame(history)

    history_path = (
        OUTPUT_DIR /
        f"history_seed_{seed}.csv"
    )

    history_df.to_csv(
        history_path,
        index=False
    )

    # --------------------------------------------------------
    # Free GPU memory before the next seed
    # --------------------------------------------------------

    del model
    del optimizer
    del scaler
    del train_loader
    del dev_loader
    del test_loader

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("\nSeed result:")
    print(
        f"Seed {seed}: "
        f"test Spearman = {test_spearman:.4f}"
    )

    print(
        f"Runtime: {elapsed_minutes:.2f} minutes"
    )

    return result

In [16]:
# ============================================================
# 16. RUN ALL 10 SEEDS
# ============================================================

all_results = []

for seed in SEEDS:

    result = run_seed(seed)

    all_results.append(result)

    # Save after EVERY seed so that a long experiment is not
    # lost if the notebook stops before all 10 runs finish.
    results_df = pd.DataFrame(all_results)

    results_df.to_csv(
        RESULTS_PATH,
        index=False
    )

    print("\nCurrent results:")
    display(results_df)

print("\n10-seed experiment complete.")


STARTING SEED 42


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



----------------------------------------------------------------------
Seed 42 | Epoch 1/4
----------------------------------------------------------------------


...

Train loss: 0.006206
Dev Spearman: 87.6915
Best checkpoint saved (epoch 4)


Evaluating:   0%|          | 0/44 [00:00<?, ?it/s]


Seed result:
Seed 51: test Spearman = 84.4511
Runtime: 7.51 minutes

Current results:


,seed,best_epoch,best_dev_spearman,test_spearman,difference_from_paper,runtime_minutes
0,42,3,87.533064,84.306180,-0.363820,7.541019
1,43,3,87.365492,83.684361,-0.985639,7.515456
2,44,4,87.333445,84.437164,-0.232836,7.503073
3,45,3,86.998394,83.834628,-0.835372,7.513634
4,46,4,87.364162,83.751215,-0.918785,7.545202
5,47,1,87.337860,83.016717,-1.653283,7.487074
6,48,4,87.497312,84.599774,-0.070226,7.537637
7,49,4,87.444190,84.336687,-0.333313,7.497972
8,50,3,87.234323,84.298234,-0.371766,7.492855
9,51,4,87.691471,84.451104,-0.218896,7.506622



10-seed experiment complete.


In [17]:
# ============================================================
# 17. 10-SEED RESULTS
# ============================================================

results_df = pd.read_csv(
    RESULTS_PATH
)

results_df = results_df.sort_values(
    "seed"
).reset_index(drop=True)

results_df

,seed,best_epoch,best_dev_spearman,test_spearman,difference_from_paper,runtime_minutes
0,42,3,87.533064,84.306180,-0.363820,7.541019
1,43,3,87.365492,83.684361,-0.985639,7.515456
2,44,4,87.333445,84.437164,-0.232836,7.503073
3,45,3,86.998394,83.834628,-0.835372,7.513634
4,46,4,87.364162,83.751215,-0.918785,7.545202
5,47,1,87.337860,83.016717,-1.653283,7.487074
6,48,4,87.497312,84.599774,-0.070226,7.537637
7,49,4,87.444190,84.336687,-0.333313,7.497972
8,50,3,87.234323,84.298234,-0.371766,7.492855
9,51,4,87.691471,84.451104,-0.218896,7.506622


In [18]:
# ============================================================
# 18. SUMMARY STATISTICS
# ============================================================

test_scores = results_df["test_spearman"]

summary = {
    "n_seeds": len(test_scores),
    "mean": test_scores.mean(),
    "std": test_scores.std(ddof=1),
    "min": test_scores.min(),
    "max": test_scores.max(),
    "paper_mean": PAPER_SCORE,
    "difference_mean_vs_paper": test_scores.mean() - PAPER_SCORE
}

summary_df = pd.DataFrame(
    [summary]
)

summary_df

,n_seeds,mean,std,min,max,paper_mean,difference_mean_vs_paper
0,10,84.071606,0.489311,83.016717,84.599774,84.67,-0.598394


In [19]:
# ============================================================
# 19. PAPER TABLE 2 COMPARISON
# ============================================================

comparison = pd.DataFrame([
    {
        "Model": "SBERT-STSb-base",
        "Paper Mean": PAPER_SCORE,
        "Our 10-Seed Mean": test_scores.mean(),
        "Our 10-Seed Std": test_scores.std(ddof=1),
        "Difference": test_scores.mean() - PAPER_SCORE
    }
])

comparison


,Model,Paper Mean,Our 10-Seed Mean,Our 10-Seed Std,Difference
0,SBERT-STSb-base,84.67,84.071606,0.489311,-0.598394


## Interpretation

The key number to compare with the paper is the **10-seed mean**, not any individual seed.

The paper reports **84.67 ± 0.19** for `SBERT-STSb-base`. This experiment should therefore be
interpreted in two stages:

1. **Per-seed results:** show how much the result varies with the random seed.
2. **10-seed mean ± standard deviation:** this is the directly comparable statistic to the
   paper's Table 2 protocol.

A deviation from 84.67 does not by itself identify the cause. Potential sources include:

- STS-B data/preprocessing differences
- unspecified STS-B implementation details
- maximum sequence length
- batch/gradient-accumulation implementation
- optimizer/training details
- random-seed variation
